In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:

df = pd.read_csv("lab_ticket_resolution.csv")

print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)

display(df.head())

In [ ]:
print("DATASET SHAPE")
print(df.shape)

print("DATA TYPES")
print(df.dtypes)

print("\nDESCRIPTIVE STATISTICS")
display(df.describe(include="all"))

print("\nMISSING VALUES")
print(df.isna().sum())

print("\nFIRST FIVE ROWS")
display(df.head())

In [ ]:

y = df["resolution_hours"]

features = [
    "complexity_score",
    "team_experience_years"
]

X = df[features]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget variable: resolution_hours")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

In [ ]:
dummy = DummyRegressor(strategy="mean")

dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

dummy_mae = mean_absolute_error(y_test, dummy_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_pred))
dummy_r2 = r2_score(y_test, dummy_pred)

print("Dummy Regressor")
print("MAE:", dummy_mae)
print("RMSE:", dummy_rmse)
print("R2:", dummy_r2)

In [ ]:
# Linear Regression

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R2:", linear_r2)

In [ ]:
# Polynomial Regression - Degrees 2 and 3

# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

poly_results = {}

for degree in [2, 3]:
    poly_model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )

    # Cross-validation using negative MAE
    cv_scores = cross_val_score(
        poly_model,
        X_train,
        y_train,
        cv=kf,
        scoring="neg_mean_absolute_error"
    )

    mean_cv_mae = -cv_scores.mean()

    # Fit on all training data
    poly_model.fit(X_train, y_train)

    # Test prediction
    poly_pred = poly_model.predict(X_test)

    test_mae = mean_absolute_error(y_test, poly_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, poly_pred))
    test_r2 = r2_score(y_test, poly_pred)

    poly_results[degree] = {
        "CV MAE": mean_cv_mae,
        "Test MAE": test_mae,
        "Test RMSE": test_rmse,
        "Test R2": test_r2
    }

    print(f"\nPolynomial Regression - Degree {degree}")
    print("Cross-validation MAE:", mean_cv_mae)
    print("Test MAE:", test_mae)
    print("Test RMSE:", test_rmse)
    print("Test R2:", test_r2)

In [ ]:
from sklearn.model_selection import GridSearchCV
# Ridge Regression with alpha selection

ridge_pipeline = make_pipeline(
    StandardScaler(),
    Ridge()
)

ridge_param_grid = {
    "ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)

ridge_model = ridge_search.best_estimator_

ridge_pred = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print("Ridge Regression")
print("Selected alpha:", ridge_search.best_params_["ridge__alpha"])
print("Best cross-validation MAE:", -ridge_search.best_score_)
print("Test MAE:", ridge_mae)
print("Test RMSE:", ridge_rmse)
print("Test R2:", ridge_r2)

In [ ]:
# Lasso Regression with alpha selection

lasso_pipeline = make_pipeline(
    StandardScaler(),
    Lasso(max_iter=20000)
)

lasso_param_grid = {
    "lasso__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
}

lasso_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)

lasso_model = lasso_search.best_estimator_

lasso_pred = lasso_model.predict(X_test)

lasso_mae = mean_absolute_error(y_test, lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))
lasso_r2 = r2_score(y_test, lasso_pred)

print("Lasso Regression")
print("Selected alpha:", lasso_search.best_params_["lasso__alpha"])
print("Best cross-validation MAE:", -lasso_search.best_score_)
print("Test MAE:", lasso_mae)
print("Test RMSE:", lasso_rmse)
print("Test R2:", lasso_r2)

In [ ]:
# Final Model Comparison

results = pd.DataFrame({
    "Model": [
        "Dummy Regressor",
        "Linear Regression",
        "Polynomial Regression (Degree 2)",
        "Polynomial Regression (Degree 3)",
        "Ridge Regression",
        "Lasso Regression"
    ],
    "MAE": [
        dummy_mae,
        linear_mae,
        poly_results[2]["Test MAE"],
        poly_results[3]["Test MAE"],
        ridge_mae,
        lasso_mae
    ],
    "RMSE": [
        dummy_rmse,
        linear_rmse,
        poly_results[2]["Test RMSE"],
        poly_results[3]["Test RMSE"],
        ridge_rmse,
        lasso_rmse
    ],
    "R2": [
        dummy_r2,
        linear_r2,
        poly_results[2]["Test R2"],
        poly_results[3]["Test R2"],
        ridge_r2,
        lasso_r2
    ]
})

display(results.sort_values("MAE").reset_index(drop=True))

## Evaluation summary

The regression models were evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the coefficient of determination (R²). A Dummy Regressor was included as a baseline to establish the performance of a model that predicts the mean training-set resolution time for every ticket.

The linear, polynomial, Ridge, and Lasso models were compared using the same training and test split. Polynomial Regression with degrees 2 and 3 was evaluated using five-fold cross-validation on the training set. Ridge and Lasso were evaluated using cross-validation to select their regularization parameters.

The final comparison table presents the test-set metrics for each model. Lower MAE and RMSE indicate smaller prediction errors, while a higher R² indicates that the model explains more of the variation in the target variable.

The preferred model should be selected based on the actual results after feature selection and hyperparameter tuning. The comparison should also consider model complexity, validation performance, and the possibility of data leakage rather than relying on R² alone.

Because the dataset is synthetic and contains only 240 tickets, the results should not be interpreted as evidence that the selected model will achieve the same performance in a real software support environment.


In [ ]:
import matplotlib.pyplot as plt

# Model comparison: R²
plt.figure(figsize=(10, 5))
plt.bar(results["Model"], results["R2"])
plt.ylabel("R² Score")
plt.xlabel("Model")
plt.title("R² Score Comparison of Regression Models")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison: MAE
plt.figure(figsize=(10, 5))
plt.bar(results["Model"], results["MAE"])
plt.ylabel("Mean Absolute Error")
plt.xlabel("Model")
plt.title("MAE Comparison of Regression Models")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Model Comparison and Conclusion

The six regression approaches were compared using MAE, RMSE, and R². The Dummy Regressor provides a baseline for evaluating whether the models using ticket features offer a meaningful improvement over predicting the mean resolution time.

Linear Regression provides a simple reference model. Polynomial Regression allows nonlinear relationships to be represented, but additional polynomial terms may increase model complexity without improving generalization. Ridge and Lasso apply regularization to reduce the influence of model coefficients and help control model complexity.

The preferred model should be identified from the final test-set comparison after removing features that are unavailable at the intended prediction time and selecting the Ridge and Lasso regularization parameters using cross-validation.

### Final Conclusion

The final model should be selected based on the evidence from the corrected experiment, including test MAE, test RMSE, R², and cross-validation results. A model with slightly better test metrics should not automatically be preferred if the improvement is negligible or if its inputs would not be available in practice.

Since this dataset is synthetic and relatively small, further evaluation on realistic historical tickets would be necessary before considering deployment.

In [ ]:
# Example prediction using the best model

best_model = linear_model

# Predict the test set
final_predictions = best_model.predict(X_test)

print("First 10 actual values:")
print(y_test.iloc[:10].values)

print("\nFirst 10 predicted values:")
print(final_predictions[:10])

In [ ]:
# Final model evaluation

print("Final Model: Linear Regression")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R²:", linear_r2)

In [ ]:
# Residual plot - Linear Regression

linear_residuals = y_test - linear_pred

plt.figure(figsize=(7, 5))
plt.scatter(linear_pred, linear_residuals)

plt.axhline(0)

plt.xlabel("Predicted Resolution Hours")
plt.ylabel("Residual")
plt.title("Residual Plot - Linear Regression")

plt.show()

In [ ]:
# Residual plot - Ridge Regression

ridge_residuals = y_test - ridge_pred

plt.figure(figsize=(7, 5))
plt.scatter(ridge_pred, ridge_residuals)

plt.axhline(0)

plt.xlabel("Predicted Resolution Hours")
plt.ylabel("Residual")
plt.title("Residual Plot - Ridge Regression")

plt.show()

## Residual Plot Insights

Residual plots were examined to assess whether the regression models showed systematic prediction errors. A residual is the difference between the observed resolution time and the predicted resolution time.

Model Performance: For both Linear and Ridge Regression, the errors are evenly spread around zero with no curved patterns. A few individual tickets had larger errors, which likely means those particular tickets were just harder to predict.

Error Spread: The spread of errors stays consistent across the board, meaning the model's accuracy doesn't wildly change for longer or shorter resolution times.

Note of Caution: Because the dataset is small and artificial, we should take these diagnostic results with a grain of salt.


## Analysis Questions

### 10. Why is the mean baseline useful even though it ignores all features?

The Dummy regressor gives a simple starting point for evaluating the other models. It predicts the mean resolution time calculated from the training data for every ticket without using any features. 

Comparing the other models with this baseline helps determine whether the selected features provide useful predictive information. If a more complex model cannot improve on the baseline, its additional complexity may not be justified. The baseline also provides a reference for interpreting MAE, RMSE, and R².

### 11. Which model has the lowest MAE? Which has the lowest RMSE? Are they the same, and why might they differ?

Linear Regression has the lowest MAE, at approximately **2.59 hours**, and also the lowest RMSE, at approximately **3.22 hours**. Therefore, the same model has the lowest MAE and RMSE in this experiment.

MAE and RMSE can sometimes select different models because they measure errors differently. MAE treats errors more evenly, while RMSE gives greater importance to large errors because the errors are squared. Therefore, a model with a few very large prediction errors may have a substantially higher RMSE even if its MAE is relatively good.

### 12. Did polynomial features improve validation performance? What is the evidence of possible overfitting?

Polynomial Regression with degrees 2 and 3 was evaluated using five-fold cross-validation on the training set. The mean cross-validation MAE was recorded for each degree, and the final fitted models were evaluated on the test set.

The degree that performs best should be identified by comparing the cross-validation MAE values. A lower MAE indicates better average predictive performance under this validation procedure.

Adding polynomial features does not necessarily improve generalization. If a more complex model performs substantially better on its training data but worse during cross-validation or on the test set, that would provide evidence of possible overfitting. A difference between training and validation performance should be examined before concluding that overfitting has occurred.

The final conclusion should be based on the corrected cross-validation and test results rather than on polynomial degree alone.



### 13. How did regularization affect performance and coefficient magnitudes?
Ridge and Lasso introduce penalties on the model coefficients to control model complexity. Ridge uses an L2 penalty that generally shrinks coefficients toward zero, while Lasso uses an L1 penalty that can reduce some coefficients to exactly zero.

The strength of regularization is controlled by alpha. A larger alpha generally imposes a stronger penalty, while a smaller alpha allows the coefficients to be influenced more by the training data.
In this experiment, alpha is selected using cross-validation. The effect of regularization should be evaluated by comparing the selected alpha values and the resulting MAE, RMSE, and R² against those of ordinary Linear Regression.

Regularization is useful when it improves generalization or produces a more appropriate model, but it does not guarantee better test performance. 



### 14. Why is scaling included in the Ridge/Lasso pipeline?

Scaling is important because Ridge and Lasso penalize the size of model coefficients. If features have very different numerical scales, the regularization penalty may affect them unevenly.

StandardScaler puts the features on a comparable scale before regularization is applied. Keeping the scaler inside the pipeline also ensures that scaling is performed correctly during cross-validation without using information from the validation or test data.

### 15. Identify one potential leakage issue in the dataset and explain how you handled it.
The lines_changed feature is a potential source of data leakage because the final number of code lines changed during a fix may not be available when a support ticket is first received. The reopened feature also requires careful consideration because the reopening status may only become known after the initial resolution.

To reduce the risk of data leakage, I excluded ticket_id because it is an identifier rather than a meaningful predictive feature. I also excluded lines_changed and reopened from the main model because the objective is to predict resolution time at ticket creation. The model therefore uses complexity_score and team_experience_years, assuming these features are available at prediction time.



### 16. Would a random split be appropriate if tickets were collected over several years or came from repeated customers/teams?

A random split may not be appropriate if tickets were collected over several years because it could allow information from future periods to appear in the training data while older tickets are placed in the test data.

Similarly, if the same customers or teams appear repeatedly, a random split could place tickets from the same customer or team in both training and test sets. This could make the model appear more accurate than it would be for genuinely new customers or teams.

For time-dependent data, a time-based split would be more appropriate. For repeated customers or teams, group-based splitting or group cross-validation would provide a more realistic evaluation.

### 17. What additional tests would you perform before deploying this model?

Before deployment, I would:

1. Test the model on a larger and more realistic historical dataset.
2. Use a time-based test set if tickets are collected over time.
3. Check performance separately across different teams, customers, and ticket complexity levels.
4. Investigate tickets that produce unusually large prediction errors.
5. Check carefully for additional data leakage.
6. Confirm that all model features are actually available at prediction time.
7. Monitor model performance after deployment to detect changes in the ticket-resolution process.
8. Compare the model with a simple operational baseline to determine whether it provides useful practical value.

In [ ]:
# Observed vs Predicted - Linear Regression

plt.figure(figsize=(7, 5))
plt.scatter(y_test, linear_pred)

# Perfect prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()]
)

plt.xlabel("Observed Resolution Hours")
plt.ylabel("Predicted Resolution Hours")
plt.title("Observed vs Predicted - Linear Regression")

plt.show()

In [ ]:
# Observed vs Predicted - Ridge Regression

plt.figure(figsize=(7, 5))
plt.scatter(y_test, ridge_pred)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()]
)

plt.xlabel("Observed Resolution Hours")
plt.ylabel("Predicted Resolution Hours")
plt.title("Observed vs Predicted - Ridge Regression")

plt.show()

## Limitations and Further Testing

Several limitations should be considered before using the model in a real software support environment.

1. **Synthetic data:** The dataset is artificially generated and may not reflect the relationships, variability, and complexity found in real support tickets. Good performance on this dataset does not guarantee similar results in practice.

2. **Potential data leakage:** Features such as `lines_changed` and `reopened` may not be available when a ticket is first received. The model should only use information that is available at the intended prediction time.

3. **Limited dataset size:** The dataset contains 240 tickets. A larger dataset with more diverse tickets, teams, and resolution times would be needed to evaluate whether the model generalizes reliably.

4. **Feature availability and reliability:** The usefulness of the model depends on whether its input features are consistently available and measured accurately when predictions are made.

5. **Unusual tickets:** Some tickets may have unusually long resolution times or large prediction errors. These cases should be investigated to understand whether they represent data-quality problems or legitimate difficult cases.

6. **Changes over time:** Team composition, software tools, workloads, and support procedures may change. A model trained on historical data may become less accurate as these conditions change.

Before deployment, the model should be evaluated on realistic historical data, checked for leakage, and tested using an appropriate validation strategy. Performance should also be examined across different ticket categories and monitored over time. Further testing would be necessary to establish whether the model provides practical value in a real support environment.